In [1]:
import h5py
import numpy as np
import torch
import torch.nn as nn

import torch.functional as F

from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset, Dataset
import torchmetrics

from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

from sklearn.metrics import classification_report

In [64]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
n_classes = 6

In [46]:
def train_epoch(model, optimizer, train_loader, loss_fn, n_classes):

    loss_log = []
    acc_log = []
    f1_log = []
    
    model.train()

    acc_calc = torchmetrics.classification.Accuracy(
        num_classes=n_classes, task="multiclass"
    ).to(device)

    f1_calc = torchmetrics.classification.F1Score(
        num_classes=n_classes, task="multiclass", average="macro"
    ).to(device)

    for spec, label in train_loader:
        spec = spec.to(device).float()
        label = label.to(device).long()

        optimizer.zero_grad()
        output_log = model(spec)
        output = torch.argmax(output_log, dim=1)

        loss = loss_fn(output_log, label)
        loss_log.append(loss.item())
        
        #acc = acc_calc(output, label)
        acc_calc.update(output, label)

        #f1 = f1_calc(output, label)
        f1_calc.update(output, label)

        loss.backward()
        optimizer.step()

    acc = acc_calc.compute().item()
    f1 = f1_calc.compute().item()

    acc_calc.reset()
    f1_calc.reset()
        
    return np.mean(loss_log), acc, f1

def test(model, loader, loss_fn, n_classes):
    
    loss_log = []
    preds_log = []
    labels_log = []
    
    model.eval()
    
    acc_calc = torchmetrics.classification.Accuracy(
        num_classes=n_classes, task="multiclass"
    ).to(device)

    f1_calc = torchmetrics.classification.F1Score(
        num_classes=n_classes, task="multiclass", average="macro"
    ).to(device)

    with torch.no_grad():
        for spec, label in loader:
            spec = spec.to(device).float()
            label = label.to(device).long()

            labels_log.extend(list(label.cpu().numpy()))
            output_log = model(spec)
            output = torch.argmax(output_log, dim=1)
            preds_log.extend(list(output.cpu().numpy()))
    
            loss = loss_fn(output_log, label)
            loss_log.append(loss.item())

            #acc = acc_calc(output, label)
            acc_calc.update(output, label)
    
            #f1 = f1_calc(output, label)
            f1_calc.update(output, label)

    acc = acc_calc.compute().item()
    f1 = f1_calc.compute().item()

    acc_calc.reset()
    f1_calc.reset()
        
    return np.mean(loss_log), acc, f1, preds_log, labels_log

def train(model, optimizer, n_epoch, train_loader, val_loader, loss_fn, scheduler=None, n_classes=6):
    best_f1 = 0
    
    train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = [], [], [], [], [], []

    for epoch in tqdm(range(n_epoch)):
        train_loss, train_acc, train_f1 = train_epoch(model, optimizer, train_loader, loss_fn, n_classes)
        val_loss, val_acc, val_f1, preds_log, labels_log = test(model, val_loader, loss_fn, n_classes)
        
        train_loss_log.append(train_loss)
        train_acc_log.append(train_acc)
        train_f1_log.append(train_f1)

        val_loss_log.append(val_loss)
        val_acc_log.append(val_acc)
        val_f1_log.append(val_f1)

        print(f"Epoch {epoch}")
        print(f" train loss: {train_loss}, train acc: {train_acc}, train f1: {train_f1}")
        print(f" val loss: {val_loss}, val acc: {val_acc}, val f1: {val_f1}\n")
        print(classification_report(np.array(labels_log), np.array(preds_log)))

        if scheduler is not None:
            if isinstance(scheduler,torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_f1)
            else:
                scheduler.step()


        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "best_model.pt")

    return train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log

# MLP

In [11]:
class MLPDataset(Dataset):

    def __init__(self, path):
        self.path = path
        self.file = h5py.File(self.path, 'r')

    def __len__(self):
        return len(self.file['x_mlp'])

    def __getitem__(self, idx):

        x_mlp = self.file['x_mlp'][idx]
        label = self.file['class_mood_int'][idx]
        label = torch.tensor(label)

        x_mlp = torch.tensor(x_mlp).float()
        x = x_mlp / (torch.norm(x_mlp) + 1e-6)
    
        return x, label

In [12]:
dataset = MLPDataset('/kaggle/input/datasets/alexandra841/features-audio-embeddings/features_audio_embeddings.h5')

In [13]:
labels = dataset[:][1]

In [14]:
train_val_idx, test_idx = train_test_split(
    np.arange(len(dataset)), test_size=0.2, shuffle=True, random_state=42, stratify=labels
)

trainvalset = torch.utils.data.Subset(dataset, train_val_idx)
testset = torch.utils.data.Subset(dataset, test_idx)

In [15]:
labels_train = labels[train_val_idx]

In [16]:
torch.unique(labels_train, return_counts=True)

(tensor([0, 1, 2, 3, 4, 5], dtype=torch.int8),
 tensor([3461, 1590, 4101, 1961,  634,  735]))

In [17]:
train_idx, val_idx = train_test_split(
    np.arange(len(trainvalset)), test_size=0.2, shuffle=True, random_state=42, stratify=labels_train
)

trainset = torch.utils.data.Subset(trainvalset, train_idx)
valset = torch.utils.data.Subset(trainvalset, val_idx)

In [18]:
train_loader = DataLoader(trainset, batch_size=16, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(valset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(testset, batch_size=16, shuffle=False, num_workers=2)

In [19]:
class_weights = compute_class_weight(
        'balanced',
        classes=np.arange(n_classes),
        y=labels_train.numpy()
    )

In [20]:
class_weights

array([0.60107869, 1.30838574, 0.50727465, 1.06085331, 3.28128286,
       2.83038549])

In [61]:
weights = torch.tensor(class_weights, dtype=torch.float)
weights = torch.sqrt(weights)

In [62]:
weights

tensor([0.7753, 1.1438, 0.7122, 1.0300, 1.8114, 1.6824])

In [53]:
class_weights_tensor = torch.tensor(
    class_weights, dtype=torch.float32
).to(device)

In [63]:
class_weights_tensor_soft = torch.tensor(
    weights, dtype=torch.float32
).to(device)

/tmp/ipykernel_57/2974083217.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  class_weights_tensor_soft = torch.tensor(


In [67]:
class MLP(nn.Module):

    def __init__(self, features_in=1536, n_classes=6):

        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(features_in, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        return self.net(x)

In [68]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor).to(device)

In [69]:
model = MLP()
model = model.float()
model = model.to(device)

In [70]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

In [34]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model,
    optimizer,
    30,
    train_loader, 
    val_loader, 
    loss_fn,
    scheduler
)

  3%|▎         | 1/30 [00:11<05:28, 11.32s/it]

Epoch 0
 train loss: 1.7895794860445535, train acc: 0.28725960850715637, train f1: 0.15916284918785095
 val loss: 1.786379295549575, val acc: 0.32959550619125366, val f1: 0.08606565743684769



  7%|▋         | 2/30 [00:22<05:21, 11.47s/it]

Epoch 1
 train loss: 1.7789933152305775, train acc: 0.3259214758872986, train f1: 0.18529270589351654
 val loss: 1.7613436097552062, val acc: 0.33119744062423706, val f1: 0.16924849152565002



 10%|█         | 3/30 [00:34<05:11, 11.55s/it]

...

Epoch 27
 train loss: 1.5953340597259693, train acc: 0.35777243971824646, train f1: 0.31362074613571167
 val loss: 1.644633435899285, val acc: 0.3203844726085663, val f1: 0.2754545211791992



 97%|█████████▋| 29/30 [05:31<00:11, 11.34s/it]

Epoch 28
 train loss: 1.5986180043755434, train acc: 0.35546875, train f1: 0.30839502811431885
 val loss: 1.6444124788235708, val acc: 0.32238686084747314, val f1: 0.2777724266052246



100%|██████████| 30/30 [05:42<00:00, 11.42s/it]

Epoch 29
 train loss: 1.596515110669992, train acc: 0.36037659645080566, train f1: 0.3139559030532837
 val loss: 1.6443810470544609, val acc: 0.322787344455719, val f1: 0.2781374454498291



# RNN

In [22]:
class RNNDataset(Dataset):

    def __init__(self, path):
        self.path = path
        self.file = h5py.File(self.path, 'r')

    def __len__(self):
        return len(self.file['x_rnn'])

    def __getitem__(self, idx):

        x_rnn = self.file['x_rnn'][idx]
        label = self.file['class_mood_int'][idx]
        label = torch.tensor(label)

        x = torch.tensor(x_rnn).float()
        # по времени нормализация
        #x = x_rnn / (torch.norm(x_rnn, dim=1, keepdim=True) + 1e-6)
    
        return x, label

In [23]:
dataset_rnn = RNNDataset('/kaggle/input/datasets/alexandra841/features-audio-embeddings/features_audio_embeddings.h5')

In [24]:
train_val_idx, test_idx = train_test_split(
    np.arange(len(dataset_rnn)), test_size=0.2, shuffle=True, random_state=42, stratify=labels
)

trainvalset_rnn = torch.utils.data.Subset(dataset_rnn, train_val_idx)
testset_rnn = torch.utils.data.Subset(dataset_rnn, test_idx)

In [25]:
train_idx, val_idx = train_test_split(
    np.arange(len(trainvalset_rnn)), test_size=0.2, shuffle=True, random_state=42, stratify=labels_train
)

trainset_rnn = torch.utils.data.Subset(trainvalset_rnn, train_idx)
valset_rnn = torch.utils.data.Subset(trainvalset_rnn, val_idx)

In [26]:
train_loader_rnn = DataLoader(trainset_rnn, batch_size=16, shuffle=True, num_workers=2, drop_last=True)
val_loader_rnn = DataLoader(valset_rnn, batch_size=16, shuffle=False, num_workers=2)
test_loader_rnn = DataLoader(testset_rnn, batch_size=16, shuffle=False, num_workers=2)

In [27]:
trainset_rnn[0][0].shape

torch.Size([256, 768])

In [28]:
class RNNModel(nn.Module):

    def __init__(self, n_classes=6):

        super().__init__()

        self.norm = nn.LayerNorm(768)

        self.gru = nn.GRU(
            input_size=768,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        x = self.norm(x)

        _, h = self.gru(x)
        h = torch.cat([h[-2], h[-1]], dim=1)
        h = self.dropout(h)

        return self.fc(h)

In [26]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor).to(device)

In [27]:
model_rnn = RNNModel()
model_rnn = model_rnn.float()
model_rnn = model_rnn.to(device)

In [28]:
optimizer_rnn = torch.optim.AdamW(
    model_rnn.parameters(),
    lr=5e-5,
    weight_decay=1e-4
)

scheduler_rnn = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_rnn,
    T_max=20
)

In [29]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model_rnn,
    optimizer_rnn,
    20,
    train_loader_rnn, 
    val_loader_rnn, 
    loss_fn,
    scheduler_rnn
)

  5%|▌         | 1/20 [04:46<1:30:48, 286.76s/it]

Epoch 0
 train loss: 1.7639351687752283, train acc: 0.27483972907066345, train f1: 0.22050045430660248
 val loss: 1.711017745315649, val acc: 0.31077292561531067, val f1: 0.25380638241767883



 10%|█         | 2/20 [08:51<1:18:36, 262.03s/it]

Epoch 1
 train loss: 1.6552912627275174, train acc: 0.3385416567325592, train f1: 0.2884940505027771
 val loss: 1.6792939172428885, val acc: 0.3259911835193634, val f1: 0.26444628834724426



 15%|█▌        | 3/20 [12:56<1:12:00, 254.13s/it]

...

Epoch 17
 train loss: 0.9503251397743439, train acc: 0.5901442170143127, train f1: 0.5940336585044861
 val loss: 1.794842194789534, val acc: 0.36563876271247864, val f1: 0.31072166562080383



 95%|█████████▌| 19/20 [1:18:47<04:08, 248.22s/it]

Epoch 18
 train loss: 0.9405792329746944, train acc: 0.594651460647583, train f1: 0.5990508198738098
 val loss: 1.800107826662671, val acc: 0.3644373118877411, val f1: 0.30978941917419434



100%|██████████| 20/20 [1:23:02<00:00, 249.14s/it]

Epoch 19
 train loss: 0.9395195433440117, train acc: 0.5947515964508057, train f1: 0.5999603271484375
 val loss: 1.8000016254224596, val acc: 0.3660392463207245, val f1: 0.311570405960083



# RNN2

In [72]:
class RNNModel2(nn.Module):

    def __init__(self, n_classes=6):

        super().__init__()

        self.norm = nn.LayerNorm(768)

        self.gru = nn.GRU(
            input_size=768,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(1024, n_classes)

    def forward(self, x):
        x = self.norm(x)

        output, h = self.gru(x)

        output_mean = output.mean(dim=1)
        output_max = output.max(dim=1).values
        x = torch.cat([output_mean, output_max], dim=1)
        x = self.dropout(x)

        return self.fc(x)

In [73]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor).to(device)

In [74]:
model_rnn = RNNModel2()
model_rnn = model_rnn.float()
model_rnn = model_rnn.to(device)

In [75]:
optimizer_rnn = torch.optim.AdamW(
    model_rnn.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler_rnn = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_rnn,
    T_max=15
)

In [76]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model_rnn,
    optimizer_rnn,
    15,
    train_loader_rnn, 
    val_loader_rnn, 
    loss_fn,
    scheduler_rnn
)

  7%|▋         | 1/15 [04:05<57:23, 245.94s/it]

Epoch 0
 train loss: 1.7474534259392664, train acc: 0.28245192766189575, train f1: 0.2406385838985443
 val loss: 1.6717910136386847, val acc: 0.2843412160873413, val f1: 0.24433007836341858

              precision    recall  f1-score   support

           0       0.49      0.13      0.20       692
           1       0.19      0.47      0.27       318
           2       0.55      0.40      0.46       821
           3       0.37      0.20      0.26       392
           4       0.08      0.21      0.12       127
           5       0.10      0.29      0.15       147

    accuracy                           0.28      2497
   macro avg       0.30      0.28      0.24      2497
weighted avg       0.41      0.28      0.30      2497



 13%|█▎        | 2/15 [08:16<53:48, 248.38s/it]

Epoch 1
 train loss: 1.619813053080669, train acc: 0.3498597741127014, train f1: 0.3089662790298462
 val loss: 1.6338333787432142, val acc: 0.3251902163028717, val f1: 0.2784990668296814

              precision    recall  f1-score   support

           0       0.52      0.33      0.40       692
           1       0.29      0.08      0.13       318
           2       0.59      0.33      0.42       821
           3       0.34      0.47      0.39       392
           4       0.09      0.27      0.14       127
           5       0.12      0.47      0.18       147

    accuracy                           0.33      2497
   macro avg       0.33      0.32      0.28      2497
weighted avg       0.44      0.33      0.35      2497



 20%|██        | 3/15 [12:27<49:59, 249.92s/it]

...

Epoch 12
 train loss: 0.5893832828419713, train acc: 0.7328726053237915, train f1: 0.7561389207839966
 val loss: 2.177632006822498, val acc: 0.4313175678253174, val f1: 0.32291826605796814

              precision    recall  f1-score   support

           0       0.44      0.58      0.50       692
           1       0.26      0.22      0.24       318
           2       0.54      0.52      0.53       821
           3       0.46      0.35      0.40       392
           4       0.11      0.08      0.09       127
           5       0.20      0.15      0.17       147

    accuracy                           0.43      2497
   macro avg       0.34      0.32      0.32      2497
weighted avg       0.42      0.43      0.42      2497



 93%|█████████▎| 14/15 [58:02<04:07, 247.39s/it]

Epoch 13
 train loss: 0.5149259730600394, train acc: 0.7628205418586731, train f1: 0.7886403799057007
 val loss: 2.14477303182813, val acc: 0.4148978888988495, val f1: 0.32853594422340393

              precision    recall  f1-score   support

           0       0.46      0.53      0.49       692
           1       0.24      0.24      0.24       318
           2       0.56      0.48      0.52       821
           3       0.43      0.40      0.41       392
           4       0.12      0.13      0.12       127
           5       0.16      0.22      0.19       147

    accuracy                           0.41      2497
   macro avg       0.33      0.33      0.33      2497
weighted avg       0.43      0.41      0.42      2497



100%|██████████| 15/15 [1:02:10<00:00, 248.68s/it]

Epoch 14
 train loss: 0.44902212645571965, train acc: 0.7918669581413269, train f1: 0.8193062543869019
 val loss: 2.2816306314176056, val acc: 0.428113728761673, val f1: 0.33449435234069824

              precision    recall  f1-score   support

           0       0.46      0.57      0.51       692
           1       0.24      0.22      0.23       318
           2       0.58      0.48      0.52       821
           3       0.44      0.42      0.43       392
           4       0.14      0.11      0.12       127
           5       0.17      0.22      0.19       147

    accuracy                           0.43      2497
   macro avg       0.34      0.34      0.33      2497
weighted avg       0.43      0.43      0.43      2497

